# PMC Miner
DOI-based mining the paper data through PMC: DOIs from `notebook/test_dois.csv`.

In [1]:
from pathlib import Path
import json

from pmc_miner import SearchBasedMiner, DOIBasedMiner
from pmc_miner.utils.logging import setup_logging

setup_logging()

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
print("notebook dir :", NOTEBOOK_DIR)
print("project root :", PROJECT_ROOT)

notebook dir : /Users/yaochenr/project/pmc_data_mining/notebook
project root : /Users/yaochenr/project/pmc_data_mining


## DOI-based mining

By using DOI-based mining, you only need to specify the DOI list. For papers not in PMC open access, they will be skipped and logged to `/dois_notin_pmc.txt`.

In [2]:
doi_csv = '/Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv'
doi_out = NOTEBOOK_DIR / "doi_mining_test"

doi_miner = DOIBasedMiner(
    output_dir=str(doi_out),
    paper_type="test", # a label
    download_images=True,
)

doi_stats = doi_miner.mine_from_csv(str(doi_csv))
doi_stats

2026-05-21 18:56:29,077 - INFO - Loaded 10 DOIs from /Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv
2026-05-21 18:56:29,078 - INFO - DOI-based mining (test): 10 DOIs
2026-05-21 18:56:29,078 - INFO - [1/10] 10.1002/prp2.281
2026-05-21 18:56:29,872 - INFO - Found PMC ID PMC5461651 for DOI 10.1002/prp2.281 (Open Access)
2026-05-21 18:56:30,384 - INFO - Retrieved metadata for 1 papers
2026-05-21 18:56:31,257 - INFO - Successfully retrieved XML for PMC5461651
2026-05-21 18:56:31,288 - INFO - Saved metadata for PMCPMC5461651
2026-05-21 18:56:31,289 - INFO - Saved XML for PMCPMC5461651
2026-05-21 18:56:31,292 - INFO - Saved processed content for PMCPMC5461651
2026-05-21 18:56:31,292 - INFO - Listing S3 figure objects for PMC5461651
2026-05-21 18:56:32,156 - INFO - Found 13 figure images for PMC5461651
2026-05-21 18:56:32,338 - INFO - Downloaded: prp2281-fig-0001_PRP2-5-e00281-g001.jpg
2026-05-21 18:56:33,013 - INFO - Downloaded: prp2281-fig-0002_PRP2-5-e00281-g002.jpg
2026-05-

{'session_start_time': '2026-05-21T18:56:29.078304',
 'completion_time': '2026-05-21T18:58:09.973536',
 'source_file': '/Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv',
 'output_directory': '/Users/yaochenr/project/pmc_data_mining/notebook/doi_mining_test',
 'paper_type': 'test',
 'statistics': {'total_dois': 10,
  'found_pmc_ids': 10,
  'successfully_processed': 10,
  'failed_processing': 0,
  'not_in_pmc': 0,
  'not_open_access': 0,
  'already_processed': 0},
 'successful_papers': [{'doi': '10.1002/prp2.281', 'pmc_id': 'PMC5461651'},
  {'doi': '10.3390/metabo13040576', 'pmc_id': 'PMC10141538'},
  {'doi': '10.3390/molecules24010071', 'pmc_id': 'PMC6337141'},
  {'doi': '10.3389/fphar.2023.1207928', 'pmc_id': 'PMC10308081'},
  {'doi': '10.1021/tx500031p', 'pmc_id': 'PMC4028327'},
  {'doi': '10.1021/acs.chemrestox.3c00192', 'pmc_id': 'PMC10583834'},
  {'doi': '10.1073/pnas.2518096122', 'pmc_id': 'PMC12745818'},
  {'doi': '10.1155/2021/8434204', 'pmc_id': 'PMC8166468'},
  

In [3]:
pmc_dirs = sorted(doi_out.glob("PMC*"))
print(f"Stored {len(pmc_dirs)} papers:")
for p in pmc_dirs:
    meta = json.loads((p / "metadata.json").read_text())
    n_images = len(list((p / "images").glob("*.jpg"))) if (p / "images").exists() else 0
    n_supp = len(list((p / "supplementary").iterdir())) if (p / "supplementary").exists() else 0
    print(f"  {p.name}  doi={meta.get('doi')}  images={n_images}  supp_files={n_supp}")

skipped = doi_out / "dois_notin_pmc.txt"
if skipped.exists():
    print("\nSkipped (not in PMC OA):")
    print(skipped.read_text())

Stored 10 papers:
  PMC10141538  doi=10.3390/metabo13040576  images=8  supp_files=2
  PMC10308081  doi=10.3389/fphar.2023.1207928  images=14  supp_files=1
  PMC10444680  doi=10.1007/s00216-023-04815-3  images=9  supp_files=2
  PMC10583834  doi=10.1021/acs.chemrestox.3c00192  images=11  supp_files=2
  PMC12745818  doi=10.1073/pnas.2518096122  images=3  supp_files=1
  PMC4028327  doi=10.1021/tx500031p  images=10  supp_files=2
  PMC5461651  doi=10.1002/prp2.281  images=13  supp_files=0
  PMC6321230  doi=10.3390/pharmaceutics10040178  images=5  supp_files=2
  PMC6337141  doi=10.3390/molecules24010071  images=7  supp_files=2
  PMC8166468  doi=10.1155/2021/8434204  images=9  supp_files=0
